# Fake News Detector — Final Working Pipeline

This notebook is configured for the **exact datasets supplied with this project**:

- `Fake.csv.zip`
- `True.csv.zip`

Each ZIP contains its corresponding CSV.

### Complete pipeline

**ZIP datasets → extraction → validation → title + text → cleaning → stratified split → TF-IDF → LinearSVC → evaluation → save model → new-news prediction**

The pipeline is deliberately self-contained so it does not depend on Kaggle paths or manually installed SpaCy language models.


In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import zipfile
import re
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TEST_SIZE = 0.20
MODEL_PATH = Path("fake_news_detector.joblib")

print("All libraries imported successfully.")


## 2. Locate the supplied ZIP datasets

The notebook expects the two uploaded files to be placed in the same folder as the notebook:

```text
Fake.csv.zip
True.csv.zip
```

If you move the files, change `DATA_DIR`.


In [ ]:
# ============================================================
# 2. DATASET PATHS
# ============================================================

DATA_DIR = Path(".")

FAKE_ZIP = DATA_DIR / "Fake.csv.zip"
TRUE_ZIP = DATA_DIR / "True.csv.zip"

if not FAKE_ZIP.exists():
    raise FileNotFoundError(
        f"Missing dataset: {FAKE_ZIP.resolve()}\n"
        "Place Fake.csv.zip beside this notebook."
    )

if not TRUE_ZIP.exists():
    raise FileNotFoundError(
        f"Missing dataset: {TRUE_ZIP.resolve()}\n"
        "Place True.csv.zip beside this notebook."
    )

print("Fake dataset:", FAKE_ZIP.resolve())
print("True dataset:", TRUE_ZIP.resolve())


In [ ]:
# ============================================================
# 3. EXTRACT ZIP FILES
# ============================================================

EXTRACT_DIR = DATA_DIR / "extracted_data"
EXTRACT_DIR.mkdir(exist_ok=True)

def extract_csv_from_zip(zip_path, extract_dir):
    with zipfile.ZipFile(zip_path, "r") as z:
        csv_names = [
            name for name in z.namelist()
            if name.lower().endswith(".csv") and not name.endswith("/")
        ]

        if not csv_names:
            raise ValueError(f"No CSV file found inside {zip_path.name}")

        # These supplied ZIPs contain one CSV each.
        z.extract(csv_names[0], extract_dir)
        return extract_dir / csv_names[0]

FAKE_CSV = extract_csv_from_zip(FAKE_ZIP, EXTRACT_DIR)
TRUE_CSV = extract_csv_from_zip(TRUE_ZIP, EXTRACT_DIR)

print("Extracted:")
print(" ", FAKE_CSV.resolve())
print(" ", TRUE_CSV.resolve())


In [ ]:
# ============================================================
# 4. LOAD DATA
# ============================================================

fake = pd.read_csv(FAKE_CSV)
true = pd.read_csv(TRUE_CSV)

required_columns = {"title", "text", "subject", "date"}

for name, frame in [("Fake.csv", fake), ("True.csv", true)]:
    missing = required_columns - set(frame.columns)
    if missing:
        raise ValueError(
            f"{name} is missing columns: {sorted(missing)}"
        )

# Labels:
# 1 = Fake
# 0 = True
fake["category"] = 1
true["category"] = 0

df = pd.concat([fake, true], ignore_index=True)

print(f"Fake articles : {len(fake):,}")
print(f"True articles : {len(true):,}")
print(f"Total articles: {len(df):,}")
print("\nColumns:", df.columns.tolist())


In [ ]:
# ============================================================
# 5. DATA VALIDATION
# ============================================================

print("Missing values:")
print(df[["title", "text", "subject", "date"]].isna().sum())

print("\nExact duplicate rows:", df.duplicated().sum())

print("\nClass distribution:")
print(
    df["category"]
    .value_counts()
    .sort_index()
    .rename(index={0: "True", 1: "Fake"})
)


In [ ]:
# ============================================================
# 6. CREATE MODEL INPUT TEXT
# ============================================================

# Title is useful information, so combine it with article text.
df["title"] = df["title"].fillna("").astype(str)
df["text"] = df["text"].fillna("").astype(str)

df["content"] = (
    df["title"].str.strip()
    + " "
    + df["text"].str.strip()
).str.strip()

# Remove articles with no usable text.
df = df.loc[
    df["content"].str.len() > 0,
    ["content", "category"]
].copy()

# Remove duplicate articles before splitting.
df = (
    df.drop_duplicates(subset=["content"])
      .reset_index(drop=True)
)

print("Usable unique articles:", f"{len(df):,}")
print(df.head(3))


## 7. Text cleaning

We use lightweight normalization instead of SpaCy/NLTK because the original approach required an external SpaCy model and NLTK resources.

The cleaning function is also reused when a user submits new news to the model.

Steps:

1. lowercase
2. remove URLs
3. remove HTML
4. remove punctuation/special characters
5. normalize whitespace


In [ ]:
# ============================================================
# 7. TEXT CLEANING
# ============================================================

def clean_text(text):
    text = str(text).lower()

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Remove HTML
    text = re.sub(r"<[^>]+>", " ", text)

    # Keep letters, numbers and whitespace
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

df["content"] = df["content"].map(clean_text)

# Remove anything that became empty.
df = df.loc[df["content"].str.len() > 0].reset_index(drop=True)

print("Dataset after cleaning:", df.shape)


In [ ]:
# ============================================================
# 8. CLASS DISTRIBUTION
# ============================================================

counts = df["category"].value_counts().sort_index()

plt.figure(figsize=(7, 4))
plt.bar(
    ["True News", "Fake News"],
    [counts.get(0, 0), counts.get(1, 0)]
)
plt.title("Fake vs True News")
plt.ylabel("Number of Articles")
plt.tight_layout()
plt.show()


## 9. Train/Test Split

The data is split **before TF-IDF is fitted**.

This prevents the test set from influencing the vocabulary or IDF statistics.

`stratify=y` keeps the Fake/True distribution balanced between train and test sets.


In [ ]:
# ============================================================
# 9. TRAIN / TEST SPLIT
# ============================================================

X = df["content"]
y = df["category"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training samples:", f"{len(X_train):,}")
print("Testing samples :", f"{len(X_test):,}")


## 10. Final ML Pipeline

```text
Raw article
    ↓
TF-IDF Vectorization
    ↓
Linear Support Vector Classifier
    ↓
Fake / True
```

The TF-IDF vectorizer and classifier are stored together inside one `Pipeline`.

This is important for deployment because inference uses exactly the same transformation learned during training.


In [ ]:
# ============================================================
# 10. BUILD PIPELINE
# ============================================================

model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2),
            min_df=3,
            max_df=0.95,
            sublinear_tf=True,
            max_features=100_000,
            dtype=np.float32
        )
    ),
    (
        "classifier",
        LinearSVC(
            C=1.0,
            class_weight="balanced",
            random_state=RANDOM_STATE
        )
    )
])

print(model)


In [ ]:
# ============================================================
# 11. TRAIN
# ============================================================

model.fit(X_train, y_train)

print("Model training completed successfully.")


In [ ]:
# ============================================================
# 12. EVALUATION
# ============================================================

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy * 100:.2f}%\n")

print("Classification Report:")
print(
    classification_report(
        y_test,
        predictions,
        target_names=["True News", "Fake News"],
        digits=4
    )
)

cm = confusion_matrix(y_test, predictions)

print("Confusion Matrix:")
print(cm)


In [ ]:
# ============================================================
# 13. CONFUSION MATRIX
# ============================================================

fig, ax = plt.subplots(figsize=(6, 5))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["True", "Fake"]
)

disp.plot(ax=ax, values_format="d")
ax.set_title("Fake News Detector — Confusion Matrix")

plt.tight_layout()
plt.show()


## 14. Predict new news

`LinearSVC` does not provide probabilities by default, so the returned `decision_score` is a **decision-function score**, not a probability.

- positive score → Fake class
- negative score → True class

Do not call it a percentage confidence.


In [ ]:
# ============================================================
# 14. PREDICTION FUNCTION
# ============================================================

def predict_news(title, text):
    content = clean_text(f"{title} {text}")

    if not content:
        raise ValueError("News title/text cannot be empty.")

    prediction = int(model.predict([content])[0])
    score = float(model.decision_function([content])[0])

    return {
        "prediction": "FAKE NEWS" if prediction == 1 else "TRUE NEWS",
        "label": prediction,
        "decision_score": score
    }


# Example
result = predict_news(
    "Example news headline",
    "This is an example article used to test the trained classifier."
)

print(result)


In [ ]:
# ============================================================
# 15. SAVE COMPLETE PIPELINE
# ============================================================

joblib.dump(model, MODEL_PATH)

print("Model saved successfully:")
print(MODEL_PATH.resolve())


In [ ]:
# ============================================================
# 16. LOAD SAVED PIPELINE
# ============================================================

loaded_model = joblib.load(MODEL_PATH)

def predict_with_loaded_model(title, text):
    content = clean_text(f"{title} {text}")

    if not content:
        raise ValueError("News title/text cannot be empty.")

    prediction = int(loaded_model.predict([content])[0])
    score = float(loaded_model.decision_function([content])[0])

    return {
        "prediction": "FAKE NEWS" if prediction == 1 else "TRUE NEWS",
        "label": prediction,
        "decision_score": score
    }

print(
    predict_with_loaded_model(
        "Test headline",
        "Test article text."
    )
)


# Final Production Flow

```text
                 ┌──────────────────┐
                 │ Fake.csv.zip     │
                 │ True.csv.zip     │
                 └────────┬─────────┘
                          ↓
                 ┌──────────────────┐
                 │ Extract CSV      │
                 └────────┬─────────┘
                          ↓
                 ┌──────────────────┐
                 │ Validate Data    │
                 └────────┬─────────┘
                          ↓
                 ┌──────────────────┐
                 │ Title + Text     │
                 └────────┬─────────┘
                          ↓
                 ┌──────────────────┐
                 │ Text Cleaning    │
                 └────────┬─────────┘
                          ↓
                 ┌──────────────────┐
                 │ Train/Test Split │
                 └────────┬─────────┘
                          ↓
              ┌────────────────────────┐
              │      ML PIPELINE       │
              │                        │
              │  TF-IDF → LinearSVC    │
              └────────────┬───────────┘
                           ↓
                 ┌──────────────────┐
                 │ Evaluation       │
                 │ Accuracy         │
                 │ Precision/Recall │
                 │ F1 Score         │
                 │ Confusion Matrix │
                 └────────┬─────────┘
                          ↓
                 ┌──────────────────┐
                 │ joblib model     │
                 └────────┬─────────┘
                          ↓
                 ┌──────────────────┐
                 │ FastAPI / Flask  │
                 │      /predict    │
                 └──────────────────┘
```

## Important

This model detects patterns learned from this dataset. It should **not** be presented as a factual verification system. A high test score means it classifies examples from this dataset well; it does not guarantee that an arbitrary real-world article is factually true or false.
